In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LeakyReLU, BatchNormalization
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv('creditcard.csv')  # replace with the correct path to your CSV file

# Preprocess the data
# Assuming 'Time' column is not required and 'Amount' needs to be scaled
df['Amount'] = StandardScaler().fit_transform(df['Amount'].values.reshape(-1, 1))
X = df.drop(['Class', 'Time'], axis=1)  # dropping 'Time' if not required
y = df['Class']

# Define the dimension of the latent space
latent_dim = 100  # for generating synthetic samples

# Build the Generator
def build_generator(latent_dim):
    model = Sequential()

    model.add(Dense(128, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(X.shape[1], activation='tanh'))  # Output layer size must match the number of input features

    return model

# Build the Discriminator
def build_discriminator(data_shape):
    model = Sequential()

    model.add(Dense(512, input_dim=data_shape))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(128))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5), metrics=['accuracy'])
    return model

# GAN Utility Functions
def get_real_samples(batch_size):
    idx = np.random.randint(0, X.shape[0], batch_size)
    real_samples = X.iloc[idx].values
    real_labels = np.ones((batch_size, 1))  # Label for real samples is 1
    return real_samples, real_labels

def generate_latent_points(batch_size):
    return np.random.normal(0, 1, (batch_size, latent_dim))

def generate_fake_samples(generator, batch_size):
    latent_points = generate_latent_points(batch_size)
    fake_samples = generator.predict(latent_points)
    fake_labels = np.zeros((batch_size, 1))  # Label for fake samples is 0
    return fake_samples, fake_labels

# Assemble the GAN
generator = build_generator(latent_dim)
discriminator = build_discriminator(X.shape[1])
discriminator.trainable = False  # Make sure only the generator is trained within the GAN model
gan_input = Input(shape=(latent_dim,))
fake_samples = generator(gan_input)
gan_output = discriminator(fake_samples)
gan = Model(gan_input, gan_output)
gan.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))

# Training the GAN
def train_gan(epochs, batch_size):
    for epoch in range(epochs):
        # Get randomly selected 'real' samples
        real_samples, real_labels = get_real_samples(batch_size // 2)
        # Generate 'fake' samples
        fake_samples, fake_labels = generate_fake_samples(generator, batch_size // 2)
        # Train the discriminator
        d_loss_real = discriminator.train_on_batch(real_samples, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_samples, fake_labels)
        # Prepare points in the latent space as input for the generator
        latent_points = generate_latent_points(batch_size)
        # Labels for fake samples are 1 to fool the discriminator
        fake_labels = np.ones((batch_size, 1))
        # Train the GAN on the latent points
        g_loss = gan.train_on_batch(latent_points, fake_labels)

        # Output training progress
        print(f"Epoch {epoch+1}/{epochs} | D Loss Real: {d_loss_real[0]}, D Loss Fake: {d_loss_fake[0]}, G Loss: {g_loss}")

# Train GAN
train_gan(epochs=10000, batch_size=32)  # Adjust the number of epochs and batch size according to your requirements


1/1 [==============================] - 1s 1s/step


Epoch 1/10000 | D Loss Real: 0.5899489521980286, D Loss Fake: 0.7530322074890137, G Loss: 0.569890022277832
1/1 [==============================] - 0s 44ms/step
Epoch 2/10000 | D Loss Real: 0.5321063995361328, D Loss Fake: 0.7557721734046936, G Loss: 0.5482107400894165
1/1 [==============================] - 0s 118ms/step
Epoch 3/10000 | D Loss Real: 0.5046788454055786, D Loss Fake: 0.7749607563018799, G Loss: 0.5291659235954285
1/1 [==============================] - 0s 63ms/step
Epoch 4/10000 | D Loss Real: 0.4779772460460663, D Loss Fake: 0.8191459774971008, G Loss: 0.4991912245750427
1/1 [==============================] - 0s 55ms/step
Epoch 5/10000 | D Loss Real: 0.4798276424407959, D Loss Fake: 0.8316313028335571, G Loss: 0.5008749961853027
1/1 [==============================] - 0s 49ms/step
Epoch 6/10000 | D Loss Real: 0.4725768566131592, D Loss Fake: 0.8585072755813599, G Loss: 0.49136969447135925
1/1 [===========================

In [4]:
# Assuming 'generator' is your pre-trained generator model from the GAN

# Number of synthetic samples to create
# This should be based on how much you want to balance the classes
num_synthetic_samples = 10000  # example number

# Generate synthetic transaction data
latent_points = generate_latent_points(num_synthetic_samples)
synthetic_data = generator.predict(latent_points)

# Convert synthetic data to a DataFrame
synthetic_data_df = pd.DataFrame(synthetic_data, columns=X.columns)

# Add a 'Class' column to the synthetic data, and set it to '1' for fraudulent transactions
synthetic_data_df['Class'] = 1  # Assuming you want to generate data for the minority class

313/313 [==============================] - 1s 4ms/step


In [7]:
# Concatenate the synthetic data with the original data
augmented_data = pd.concat([df, synthetic_data_df])

# Shuffle the augmented dataset
augmented_data = augmented_data.sample(frac=1).reset_index(drop=True)


In [9]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
import numpy as np

# Example data (replace with your actual data)
X_train_aug = np.array([[1, 2], [np.nan, 3], [7, 6], [5, np.nan]])
X_test_aug = np.array([[1, 2], [2, np.nan], [6, 7], [np.nan, 5]])
y_train_aug = np.array([0, 1, 1, 0])

# Create an imputer object with a median filling strategy
imputer = SimpleImputer(strategy='median')

# Train on the training data
imputer.fit(X_train_aug)

# Transform both training and testing data
X_train_aug = imputer.transform(X_train_aug)
X_test_aug = imputer.transform(X_test_aug)

# Now, create and train the Logistic Regression model
lr_model = LogisticRegression(max_iter=1000)  # Increasing max_iter for convergence
lr_model.fit(X_train_aug, y_train_aug)

# Make predictions on the test data
y_test_pred_proba = lr_model.predict_proba(X_test_aug)[:, 1]

# Assuming you have true labels for the test data
y_test_true = np.array([0, 1, 1, 0])

# Calculate the Average Precision Score (AUPRC)
auprc = average_precision_score(y_test_true, y_test_pred_proba)

print("AUPRC:", auprc)


AUPRC: 0.8333333333333333


In [10]:
# Check if any NaN values are present in the synthetic data
if np.isnan(synthetic_data).any():
    synthetic_data = np.nan_to_num(synthetic_data)  # Replace NaNs with 0 or use other strategies

# Convert synthetic data to a DataFrame as before
synthetic_data_df = pd.DataFrame(synthetic_data, columns=X.columns)
synthetic_data_df['Class'] = 1  # Assuming you want to generate data for the minority class

# Proceed with concatenation and model training...


In [11]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

# Assuming generator is a pre-trained generator model from GAN
def generate_synthetic_data(generator, num_samples):
    latent_points = generate_latent_points(num_samples)
    synthetic_data = generator.predict(latent_points)
    return synthetic_data

# Assuming this is the correct path to your CSV file
df = pd.read_csv('E:/2024/July2024/Credit_Card_Gan/creditcard.csv')

# Preprocess the data
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df = df.drop(['Time'], axis=1)  # Drop 'Time' if it's not needed

# Split the real data into features and target
X_real = df.drop('Class', axis=1)
y_real = df['Class']

# Generate synthetic data (equal to the number of real instances of the minority class)
num_fraud = y_real.sum()
synthetic_fraud = generate_synthetic_data(generator, num_fraud)

# Create DataFrame for synthetic data
synthetic_df = pd.DataFrame(synthetic_fraud, columns=X_real.columns)
synthetic_df['Class'] = 1  # All synthetic instances are considered fraudulent

# Concatenate the synthetic data with the original data
augmented_df = pd.concat([df, synthetic_df])

# Handling NaN values
imputer = SimpleImputer(strategy='mean')
X_augmented = imputer.fit_transform(augmented_df.drop('Class', axis=1))
y_augmented = augmented_df['Class'].values

# Split the augmented data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_augmented, y_augmented, test_size=0.2, random_state=42)

# Retrain Logistic Regression model
lr_model = LogisticRegression(max_iter=10000)
lr_model.fit(X_train, y_train)

# Retrain Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100)
rf_model.fit(X_train, y_train)

# Retrain XGBoost Classifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Make predictions with Logistic Regression
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Make predictions with Random Forest
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Make predictions with XGBoost
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate models
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

# Calculate Average Precision Score
auprc_lr = average_precision_score(y_test, y_pred_proba_lr)
auprc_rf = average_precision_score(y_test, y_pred_proba_rf)
auprc_xgb = average_precision_score(y_test, y_pred_proba_xgb)

print(f"AUPRC (Logistic Regression): {auprc_lr}")
print(f"AUPRC (Random Forest): {auprc_rf}")
print(f"AUPRC (XGBoost): {auprc_xgb}")

16/16 [==============================] - 0s 4ms/step
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56861
           1       0.84      0.26      0.40       199

    accuracy                           1.00     57060
   macro avg       0.92      0.63      0.70     57060
weighted avg       1.00      1.00      1.00     57060

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56861
           1       0.97      0.79      0.87       199

    accuracy                           1.00     57060
   macro avg       0.98      0.89      0.93     57060
weighted avg       1.00      1.00      1.00     57060

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56861
           1       0.98      0.88      0.93       199

    accuracy      